# אימון גלאי אבטחה (security.pt): אקדח, סכין, אש, עשן — Google Colab

לפני שמריצים: **Runtime → Change runtime type → T4 GPU**.

המחברת מורידה את שני הדאטהסטים ישירות לקולאב (אקדחים 1.5 GB ממצלמות אבטחה אמיתיות, D-Fire 3.1 GB), מוסיפה את הדאטהסטים הנוספים שלכן (סכינים), בונה דאטהסט אחד, מאמנת YOLOv8s ומורידה `best.pt`.

**סכינים:** בדאטהסט של מצלמות האבטחה יש רק 210 תיבות של סכין, ולכן `security.pt` הראשון כמעט לא מזהה סכינים. `knives.zip` (נבנה במחשב עם `train/pack_exports.py` משני דאטהסטים של Roboflow Universe: 2,926 תמונות אימון + 585 validation, 3,600 תיבות) מתקן את זה. שימו אותו ב-Google Drive בתיקייה בשם **`security_exports`** (270 MB; העלאה ישירה לקולאב איטית).

מומלץ טוקן של Hugging Face ב-Secrets (`HF_TOKEN`, Notebook access דלוק), אחרת ההורדה עלולה להיחסם זמנית.

In [ ]:
# 1. setup
!pip -q install ultralytics huggingface_hub pyarrow
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — switch runtime to GPU')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF token: ok')
except Exception:
    print('no HF_TOKEN secret — continuing without it')

In [ ]:
# 2. the dataset scripts from the repo
import urllib.request, sys
BASE = 'https://raw.githubusercontent.com/Talyaavraham/final_project/main/backend/train/'
for f in ('night_dataset.py', 'security_dataset.py'):
    urllib.request.urlretrieve(BASE + f, '/content/' + f); print('got', f)
sys.path.insert(0, '/content')

In [ ]:
# 3. download the sources (~4.6 GB; a few minutes on Colab)
from huggingface_hub import hf_hub_download, snapshot_download
os.makedirs('/content/backend/eval/data/weapons', exist_ok=True)
hf_hub_download('jsalazar/US-Real-time-gun-detection-in-CCTV-An-open-problem-dataset', 'weapons_images_2fps.zip',
                repo_type='dataset', local_dir='/content/backend/eval/data/weapons')
snapshot_download('badsaarow/d-fire', repo_type='dataset', local_dir='/content/backend/train/security/dfire',
                  allow_patterns=['data/*.parquet'])
print('sources ready')

In [ ]:
# 4. extra exports (zips in the Roboflow "YOLOv8" layout): knives.zip from train/pack_exports.py, your own labelled frames.
#    Fastest: put the zips in Google Drive, in a folder named security_exports (approve the Drive prompt).
#    Without Drive an upload dialog opens (slow for 270 MB; Cancel = train without extras).
import glob, shutil, zipfile
zips = []
try:
    from google.colab import drive
    drive.mount('/content/drive')
    zips = sorted(glob.glob('/content/drive/MyDrive/security_exports/*.zip'))
except Exception as e:
    print('Drive not mounted:', e)
if not zips:
    from google.colab import files
    zips = list(files.upload())
own = []
for i, name in enumerate(zips):
    root = f'/content/own{i}'
    shutil.rmtree(root, ignore_errors=True)
    zipfile.ZipFile(name).extractall(root)
    if not os.path.exists(root + '/data.yaml'):
        subs = [d for d in os.listdir(root) if os.path.isdir(f'{root}/{d}')]
        root = f'{root}/{subs[0]}' if subs else root
    own.append(root); print('export:', name, '->', root)
print('exports:', own or 'NONE — the knife class will stay as weak as before')

In [ ]:
# 5. build the dataset (classes: gun, knife, fire, smoke)
import security_dataset as sd
sd.WEAPONS_ZIP = '/content/backend/eval/data/weapons/weapons_images_2fps.zip'
sd.DFIRE_DIR = '/content/backend/train/security/dfire/data'
w = sd.Writer('/content/security/dataset')
sd.add_weapons(w); sd.add_dfire(w)
base_val = list(w.files['val'])            # the validation images of the first security.pt (2026-11-10)
for d in own: sd.add_own(w, d)             # a big export is used once, a few own frames 4 times
w.yaml(); w.yaml('data_base.yaml', base_val)
print(w.n, w.boxes, '| base validation images:', len(base_val))

In [ ]:
# 6. train. yolov8s, imgsz 800: ~3.5 min/epoch on a T4 -> 25 epochs ≈ 1.5-2 h. In a hurry: epochs=12.
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/security/dataset/data.yaml', epochs=25, imgsz=800, batch=16, device=0,
            project='/content/runs', name='security', exist_ok=True, seed=0,
            close_mosaic=5, scale=0.5, patience=8)

In [ ]:
# 7. metrics per class. "all validation" includes the new knife photos (easier than CCTV, and frames of the same
#    clips are in training: flattering). "base" = weapons CCTV + D-Fire only, the same images as on 2026-11-10,
#    when the first security.pt got: gun 0.69, knife 0.35, fire 0.73, smoke 0.84 -> gun / fire / smoke must not drop.
m = YOLO('/content/runs/security/weights/best.pt')
for title, yaml in (('all validation', 'data.yaml'), ('base (comparable with 2026-11-10)', 'data_base.yaml')):
    r = m.val(data='/content/security/dataset/' + yaml, imgsz=800, plots=False, verbose=False)
    print('\n' + title)
    for k, c in enumerate(r.ap_class_index):
        print(f'  {r.names[int(c)]:8s} P={r.box.p[k]:.3f} R={r.box.r[k]:.3f} mAP50={r.box.ap50[k]:.3f}  mAP50-95={r.box.ap[k]:.3f}')
    print('  all      mAP50=%.3f' % r.box.map50)

In [ ]:
# 8. download best.pt  ->  save it as backend/security.pt (keep the old one as security_v1.pt to compare)
from google.colab import files
files.download('/content/runs/security/weights/best.pt')

## חזרה למחשב

העתיקו לצ'אט את הפלט של תא 7 (המספרים לכל מחלקה), ואז:

```powershell
cd backend
copy security.pt security_v1.pt          # הגיבוי של המודל הקודם
copy $env:USERPROFILE\Downloads\best.pt security.pt
$env:YOLO_MODEL = "yolov8n.pt,game.pt,night.pt"
$env:EXTRA_MODELS = "security.pt"
$env:IMGSZ = "800"
.\venv\Scripts\python -m uvicorn main:app --port 8000
```

`EXTRA_MODELS` = מודלים שרצים **בנוסף** למודל שנבחר לסרטון (הגלאי הראשי מוצא אנשים ורכבים, security.pt מוצא נשק ואש באותם פריימים). סרטונים שכבר אונדקסו צריך להעלות מחדש כדי שהמודל החדש ירוץ עליהם.